## Exploratory Data Analysis of Dataset 1 - True News
---

### Import Relevant Libraries

In [30]:
# For dataset manipulation
import numpy as np
import pandas as pd
# For punctuation count
import spacy
# For readability ease count
import textstat
# For progress bar
from tqdm.notebook import tqdm
tqdm.pandas()

### Load The spaCy English Model

In [31]:
nlp = spacy.load("en_core_web_sm")

### Custom Functions

In [32]:
# Removes whitespaces and lowers case
def whitespace_lower(component):
    return component.str.lower().str.split().str.join(' ')

# Calculate total words
def calc_word(component):
    return component.astype(str).apply(lambda x: len(x.split()))

# Calculate total punctuations
def calc_punct(component):
    doc = nlp(component)
    # For every punct found, sum increases by 1
    return sum(1 for token in doc if token.is_punct) 

# Calculate readability ease
def calc_readability(component):
    return textstat.flesch_reading_ease(component)


### Fundamental Data Analysis

In [33]:
# Read the true news dataset from Dataset 1 and store it in df_true
df_true = pd.read_csv("../../Datasets/Dataset1_true.csv")

In [34]:
# General overview
df_true_info = df_true.info()
df_true_info

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21417 entries, 0 to 21416
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    21417 non-null  object
 1   text     21417 non-null  object
 2   subject  21417 non-null  object
 3   date     21417 non-null  object
dtypes: object(4)
memory usage: 669.4+ KB


In [35]:
# Amount of unique values per column
df_true_unique_values = df_true.nunique()
df_true_unique_values

title      20826
text       21192
subject        2
date         716
dtype: int64

In [36]:
# Total number of NaN (null) values
df_true_total_null_values = df_true.isna().sum()
df_true_total_null_values 

title      0
text       0
subject    0
date       0
dtype: int64

In [37]:
# Total number of duplicate values for title
df_true_title_duplicate_count = df_true['title'].duplicated(keep='first').sum() # keep="first" ensures the first occurence of the duplicate is kept
df_true_title_duplicate_count

591

In [38]:
# Total number of duplicate values for text
df_true_text_duplicate_count = df_true['text'].duplicated(keep='first').sum()
df_true_text_duplicate_count

225

In [39]:
# Total number of combined duplicate values
df_true_combined_duplicate_count = df_true.duplicated(subset=['title', 'text']).sum()
df_true_combined_duplicate_count

220

### Drop Combined Raw Duplicates

In [40]:
df_true_no_raw_duplicates = df_true.drop_duplicates(subset=['title', 'text'], keep='first').reset_index(drop=True)

In [41]:
df_true_no_raw_duplicates.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21197 entries, 0 to 21196
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    21197 non-null  object
 1   text     21197 non-null  object
 2   subject  21197 non-null  object
 3   date     21197 non-null  object
dtypes: object(4)
memory usage: 662.5+ KB


### Dataset Rearrangement 1 (Title, Text, Label)

In [42]:
# Drop the non-required columns, "subject" and "date"
df_true_no_raw_duplicates = df_true_no_raw_duplicates.drop(columns=["subject", "date"]).reset_index(drop=True)

# Add a column, "label" where everything is 1 = True
df_true_no_raw_duplicates["label"] = 1

# Show the dataset
df_true_no_raw_duplicates

,title,text,label
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,1
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,1
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,1
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,1
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,1
...,...,...,...
21192,'Fully committed' NATO backs new U.S. approach...,BRUSSELS (Reuters) - NATO allies on Tuesday we...,1
21193,LexisNexis withdrew two products from Chinese ...,"LONDON (Reuters) - LexisNexis, a provider of l...",1
21194,Minsk cultural hub becomes haven from authorities,MINSK (Reuters) - In the shadow of disused Sov...,1
21195,Vatican upbeat on possibility of Pope Francis ...,MOSCOW (Reuters) - Vatican Secretary of State ...,1


### Word, Punctuation and Readability Ease Counts

In [43]:
# Count the number of words, punctuations and the ease in redability for title
df_true_no_raw_duplicates["title_word_count"] = calc_word(df_true_no_raw_duplicates["title"])
df_true_no_raw_duplicates["title_punct_count"] = df_true_no_raw_duplicates["title"].progress_apply(calc_punct) # .progress to trigger progress bar
df_true_no_raw_duplicates["title_readability_ease"] = df_true_no_raw_duplicates["title"].progress_apply(calc_readability)
 

  0%|          | 0/21197 [00:00<?, ?it/s]

  0%|          | 0/21197 [00:00<?, ?it/s]

In [44]:
# Count the number of words, punctuations and the ease in redability for text
df_true_no_raw_duplicates["text_word_count"] = calc_word(df_true_no_raw_duplicates["text"])
df_true_no_raw_duplicates["text_punct_count"] = df_true_no_raw_duplicates["text"].progress_apply(calc_punct)
df_true_no_raw_duplicates["text_readability_ease"] = df_true_no_raw_duplicates["text"].progress_apply(calc_readability)

  0%|          | 0/21197 [00:00<?, ?it/s]

  0%|          | 0/21197 [00:00<?, ?it/s]

### Dataset Rearrangement 2 (Added On Word, Punctuation and Readability Ease)


In [45]:
# Rearranging column order from left to right
df_true_no_raw_duplicates = df_true_no_raw_duplicates[["title", "title_word_count", "title_punct_count", "title_readability_ease", "text", "text_word_count", "text_punct_count", "text_readability_ease", "label"]]

# Show the updated dataset
df_true_no_raw_duplicates

,title,title_word_count,title_punct_count,title_readability_ease,text,text_word_count,text_punct_count,text_readability_ease,label
0,"As U.S. budget fight looms, Republicans flip t...",10,1,69.79,WASHINGTON (Reuters) - The head of a conservat...,749,121,43.12,1
1,U.S. military to accept transgender recruits o...,9,1,20.04,WASHINGTON (Reuters) - Transgender people will...,624,78,34.26,1
2,Senior U.S. Republican senator: 'Let Mr. Muell...,10,3,49.48,WASHINGTON (Reuters) - The special counsel inv...,457,54,49.45,1
3,FBI Russia probe helped by Australian diplomat...,9,2,36.96,WASHINGTON (Reuters) - Trump campaign adviser ...,376,50,43.02,1
4,Trump wants Postal Service to charge 'much mor...,11,2,68.77,SEATTLE/WASHINGTON (Reuters) - President Donal...,852,118,46.06,1
...,...,...,...,...,...,...,...,...,...
21192,'Fully committed' NATO backs new U.S. approach...,9,2,41.53,BRUSSELS (Reuters) - NATO allies on Tuesday we...,466,49,51.28,1
21193,LexisNexis withdrew two products from Chinese ...,7,0,30.53,"LONDON (Reuters) - LexisNexis, a provider of l...",125,18,33.54,1
21194,Minsk cultural hub becomes haven from authorities,7,0,30.53,MINSK (Reuters) - In the shadow of disused Sov...,320,48,51.28,1
21195,Vatican upbeat on possibility of Pope Francis ...,9,0,11.58,MOSCOW (Reuters) - Vatican Secretary of State ...,205,20,56.89,1


### Save The Updated Dataset

In [46]:
df_true_no_raw_duplicates.to_csv('../../Datasets/Dataset1_true_counts_cleaned_1.csv', index=False)

### Data Rearrangement 3 (Combine Title + Text)

In [47]:
df_true_combined = pd.read_csv("../../Datasets/Dataset1_true_counts_cleaned_1.csv")

In [48]:
df_true_combined['combined_data'] = df_true_combined['title'] + ' ' + df_true_combined['text']

In [49]:
df_true_combined = df_true_combined.drop(columns=[
    'title',
    'title_word_count', 
    'title_punct_count',
    'title_readability_ease',
    'text',
    'text_word_count', 
    'text_punct_count',
    'text_readability_ease'
]).reset_index(drop=True)

### Dataset Rearrangement 4 (Added On Combined Word, Punctuation and Readability Ease)

In [51]:
# Count the number of words, punctuations and the ease in redability for combined data
df_true_combined["combined_word_count"] = calc_word(df_true_combined["combined_data"])
df_true_combined["combined_punct_count"] = df_true_combined["combined_data"].progress_apply(calc_punct)
df_true_combined["combined_readability_ease"] = df_true_combined["combined_data"].progress_apply(calc_readability)

  0%|          | 0/21197 [00:00<?, ?it/s]

  0%|          | 0/21197 [00:00<?, ?it/s]

In [52]:
df_true_combined = df_true_combined[["combined_data", "combined_word_count", "combined_punct_count", "combined_readability_ease", "label"]]
df_true_combined

,combined_data,combined_word_count,combined_punct_count,combined_readability_ease,label
0,"As U.S. budget fight looms, Republicans flip t...",759,122,42.82,1
1,U.S. military to accept transgender recruits o...,633,79,33.95,1
2,Senior U.S. Republican senator: 'Let Mr. Muell...,467,57,49.96,1
3,FBI Russia probe helped by Australian diplomat...,385,52,42.51,1
4,Trump wants Postal Service to charge 'much mor...,863,120,45.86,1
...,...,...,...,...,...
21192,'Fully committed' NATO backs new U.S. approach...,475,51,51.68,1
21193,LexisNexis withdrew two products from Chinese ...,132,18,32.43,1
21194,Minsk cultural hub becomes haven from authorit...,327,48,50.87,1
21195,Vatican upbeat on possibility of Pope Francis ...,214,20,47.42,1


### Save the Updated Dataset

In [53]:
df_true_combined.to_csv('../../Datasets/Dataset1_true_combined_counts_cleaned_1.csv', index=False)